## Labeling of evalCrawl

In [28]:
import time
import pandas as pd
from transformers import pipeline
from pathlib import Path
import re
import html

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


In [29]:
df = pd.read_excel("evalCrawl.xlsx",header=None)
df.columns = ["comment"]
df.head()

,comment
0,I’ll believe it when the climate activists wri...
1,Yeah we’re all gonna die. \n\n\nNot cause of c...
2,I think climate change is the next big societa...
3,She could single handely end Climate Change
4,"Yeah, at this point carbon capture seems to be..."


In [30]:
def clean_text(text):
    text = str(text)
    
    text = html.unescape(text)          # fix &amp; etc
    text = re.sub(r"\s+", " ", text)    # removes \n, \t, multiple spaces
    text = text.strip()
    
    return text

In [31]:
df["comment"] = df["comment"].apply(clean_text)
df.head()

,comment
0,I’ll believe it when the climate activists wri...
1,Yeah we’re all gonna die. Not cause of climate...
2,I think climate change is the next big societa...
3,She could single handely end Climate Change
4,"Yeah, at this point carbon capture seems to be..."


In [32]:
labeling_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10672.80it/s]


In [33]:
labeling_pipeline("This product is amazing!")

[{'label': '5 stars', 'score': 0.8754845857620239}]

In [34]:
results = []

batch_size = 16  

comments = df["comment"].astype(str).tolist()

for i in range(0, len(comments), batch_size):
    batch = comments[i:i+batch_size]
    
    # truncate long text (fix your crash)
    batch = [t[:1000] for t in batch]
    
    # run model safely
    res = labeling_pipeline(batch, truncation=True)
    
    results.extend(res)
    
    print(f"Processed {i + len(batch)}/{len(comments)}")

Processed 16/2058
Processed 32/2058
Processed 48/2058
Processed 64/2058
Processed 80/2058
Processed 96/2058
Processed 112/2058
Processed 128/2058
Processed 144/2058
Processed 160/2058
Processed 176/2058
Processed 192/2058
Processed 208/2058
Processed 224/2058
Processed 240/2058
Processed 256/2058
Processed 272/2058
Processed 288/2058
Processed 304/2058
Processed 320/2058
Processed 336/2058
Processed 352/2058
Processed 368/2058
Processed 384/2058
Processed 400/2058
Processed 416/2058
Processed 432/2058
Processed 448/2058
Processed 464/2058
Processed 480/2058
Processed 496/2058
Processed 512/2058
Processed 528/2058
Processed 544/2058
Processed 560/2058
Processed 576/2058
Processed 592/2058
Processed 608/2058
Processed 624/2058
Processed 640/2058
Processed 656/2058
Processed 672/2058
Processed 688/2058
Processed 704/2058
Processed 720/2058
Processed 736/2058
Processed 752/2058
Processed 768/2058
Processed 784/2058
Processed 800/2058
Processed 816/2058
Processed 832/2058
Processed 848/2058

In [35]:
def convert_label(star_label):
    stars = int(star_label[0])  # "4 stars" → 4
    
    if stars <= 2:
        return "NEGATIVE"
    elif stars == 3:
        return "NEUTRAL"
    else:
        return "POSITIVE"

In [36]:
df["label"] = [convert_label(r["label"]) for r in results]
df["confidence"] = [r["score"] for r in results]
df.head()

,comment,label,confidence
0,I’ll believe it when the climate activists wri...,NEGATIVE,0.296156
1,Yeah we’re all gonna die. Not cause of climate...,NEGATIVE,0.458682
2,I think climate change is the next big societa...,POSITIVE,0.499384
3,She could single handely end Climate Change,NEGATIVE,0.293653
4,"Yeah, at this point carbon capture seems to be...",NEGATIVE,0.430321


In [37]:
df_clean = df[["comment", "label"]].copy()
print(df_clean["label"].value_counts())

label
NEGATIVE    1236
POSITIVE     423
NEUTRAL      399
Name: count, dtype: int64


In [38]:
df_clean.to_excel("full_labeled_dataset.xlsx", index=False)

In [39]:
pos = df[df["label"] == "POSITIVE"].sort_values(by="confidence", ascending=False)
pos_sample = pos.head(400)

In [40]:
neg = df[df["label"] == "NEGATIVE"].sort_values(by="confidence", ascending=False)
neg_sample = neg.head(400)

In [41]:
neu = df[df["label"] == "NEUTRAL"].sort_values(by="confidence", ascending=False)
neu_sample = neu.head(200)

In [42]:
df_final = pd.concat([pos_sample, neg_sample, neu_sample])
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

In [44]:
df_final[["comment", "label"]].to_excel("eval.xlsx", index=False)